In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp

# New imports for Tree-based GRN Inference
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
from tqdm import tqdm

from scAnalysis import (
    sc_io,
    preprocessing,
    quality_control,
    cell_cycle,
    batch_correction,
    dimensionality,
    clustering,
    trajectory,
    differential,
    enrichment,
    visualization,
    interactive_viz,
    imputation,
)

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [3]:
!pwd

/home/ayyuce/Desktop


In [4]:
local_h5ad_path = "resources/grn_benchmark/inference_data/nakatake_rna.h5ad"
tf_list_path = "resources/grn_benchmark/prior/tf_all.csv"

data = sc_io.read_h5ad(local_h5ad_path)
data.var.index = sc_io._make_unique(data.var.index.values)
print(f"Loaded: {data.n_obs} samples (bulk) and {data.n_vars} genes.")

data = preprocessing.filter_genes(data, min_cells=3)
preprocessing.normalize_total(data, target_sum=1e4)
preprocessing.log1p(data)


tf_all = pd.read_csv(tf_list_path, header=None)[0].tolist()
available_tfs = [tf for tf in tf_all if tf in data.var.index]
all_genes = data.var.index.tolist()
print(f"Found {len(available_tfs)} Transcription Factors in the dataset.")


X_matrix = data.X.toarray() if sp.issparse(data.X) else data.X
n_samples = X_matrix.shape[0]
n_genes = len(all_genes)

tf_indices = [all_genes.index(tf) for tf in available_tfs]
X_tf = X_matrix[:, tf_indices]

IO: Reading H5AD from 'resources/grn_benchmark/inference_data/nakatake_rna.h5ad' ...
IO: Loaded 460 cells × 25,090 genes.
Loaded: 460 samples (bulk) and 25090 genes.
filter_genes: keeping 25,090 / 25,090 genes.
Found 1876 Transcription Factors in the dataset.


In [5]:
target_variances = np.var(X_matrix, axis=0)
n_target_genes = min(5000, n_genes)
top_target_indices = np.argsort(target_variances)[::-1][:n_target_genes]

def infer_target_gene(gene_idx):
    target_gene_name = all_genes[gene_idx]
    y_target = X_matrix[:, gene_idx]
    
    if np.std(y_target) < 1e-6:
        return []

    model = LGBMRegressor(
        n_estimators=50,
        learning_rate=0.05,
        num_leaves=15,
        importance_type='gain',
        n_jobs=1,                 
        random_state=42,
        verbose=-1
    )
    
    model.fit(X_tf, y_target)
    importances = model.feature_importances_
    
    nz_indices = np.nonzero(importances)[0]
    
    local_edges = []
    for idx in nz_indices:
        tf_name = available_tfs[idx]
        weight = importances[idx]
        
        if tf_name != target_gene_name and weight > 0:
            local_edges.append((tf_name, target_gene_name, weight))
            
    local_edges.sort(key=lambda x: x[2], reverse=True)
    return local_edges[:50]

all_edges_nested = Parallel(n_jobs=-1, backend="threading", require="sharedmem")(
    delayed(infer_target_gene)(idx) for idx in tqdm(top_target_indices, desc="Training Models")
)

edges = [edge for sublist in all_edges_nested for edge in sublist]

grn_df = pd.DataFrame(edges, columns=['source', 'target', 'weight'])
grn_df = grn_df.sort_values(by='weight', ascending=False).head(50000).reset_index(drop=True)
grn_df['weight'] = grn_df['weight'].astype(str)

print(f"Total number of top edges extracted: {len(grn_df)}")

Training Models: 100%|████████████████████| 5000/5000 [1:55:11<00:00,  1.38s/it]


Total number of top edges extracted: 50000


In [6]:
output_anndata = ad.AnnData(
    X=np.empty((0, 0)),
    uns={
        "method_id": "scAnalyzer_LightGBM",
        "dataset_id": "nakatake",
        "prediction": grn_df[["source", "target", "weight"]]
    }
)

os.makedirs("output", exist_ok=True)
output_path = "output/nakatake_scAnalyzer_LightGBM.h5ad"
output_anndata.write_h5ad(output_path)

In [7]:
%cd task_grn_inference

/home/ayyuce/Desktop/task_grn_inference


/home/ayyuce/miniconda3/envs/singleCellAnalysis/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [8]:
!pwd

/home/ayyuce/Desktop/task_grn_inference


In [9]:
!rm resources/resources

In [10]:
!ln -sfn ../resources resources
!ls -l resources/grn_benchmark/inference_data/nakatake_rna.h5ad

-rw-rw-r-- 1 ayyuce ayyuce 283939072 Nov 14  2025 resources/grn_benchmark/inference_data/nakatake_rna.h5ad


In [11]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:           7.6Gi       4.2Gi       829Mi       602Mi       3.5Gi       3.4Gi
Swap:           19Gi       1.4Gi        18Gi


In [12]:
!bash scripts/prior/run_consensus.sh \
  --dataset nakatake \
  --new_model ../output/nakatake_scAnalyzer_LightGBM.h5ad

Config file generated at: src/utils/config.env
Adding new model: ../output/nakatake_scAnalyzer_LightGBM.h5ad
../output/nakatake_scAnalyzer_LightGBM.h5ad
Running consensus for Regression
Running regression consensus for dataset: nakatake
{'dataset': 'nakatake', 'evaluation_data': 'resources/grn_benchmark/inference_data/nakatake_rna.h5ad', 'regulators_consensus': 'resources/grn_benchmark/prior/regulators_consensus_nakatake.json', 'predictions': ['../output/nakatake_scAnalyzer_LightGBM.h5ad']}
Original net shape: (50000, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (50000, 3)
Network shape applying max_n_links: (50000, 3)
Sparsity of ../output/nakatake_scAnalyzer_LightGBM.h5ad: 0.999920572904463
Running consensus for ws distance
Skipping dataset: nakatake


In [13]:
!env JOBLIB_TEMP_FOLDER=../joblib_tmp MPLBACKEND=agg bash src/metrics/all_metrics/run_local.sh \
  --dataset nakatake \
  --prediction ../output/nakatake_scAnalyzer_LightGBM.h5ad \
  --score ../output/nakatake_score_LightGBM.h5ad \
  --num_workers 4

Layer is set to: lognorm
Regression type is set to: ridge
Number of workers is set to: 4
Dataset is set to: nakatake
Prediction file is set to: ../output/nakatake_scAnalyzer_LightGBM.h5ad
Score file is set to: ../output/nakatake_score_LightGBM.h5ad
Method id: scAnalyzer_LightGBM, Dataset id: nakatake
Computing metrics for dataset nakatake: ['regression', 'gs_recovery', 'vc']
Computing metric: regression
Layer lognorm not found, using X_norm instead
Evaluating 25090 genes (consensus data available for all)
Original net shape: (50000, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (50000, 3)
Network shape applying max_n_links: (50000, 3)
Static approach (theta=r_precision):
Static approach (theta=r_recall):
Raw approach (no theta):
theta    r2_raw  r_precision  r_recall
0      0.239094     0.236946  0.236946
Computing metric: gs_recovery
Original net shape: (50000, 3)
Supplementary columns for grouping: []
Network shape after cleaning: (50000, 3)
Network shape ap

In [14]:
import anndata as ad
import pandas as pd
from IPython.display import display

score_data = ad.read_h5ad("../output/nakatake_score_LightGBM.h5ad")

metric_ids = score_data.uns['metric_ids']
metric_values = score_data.uns['metric_values']

df_scores = pd.DataFrame({
    'Metric': metric_ids,
    'Score (scAnalyzer_LightGBM)': metric_values
})

df_scores = df_scores.sort_values(by='Score (scAnalyzer_LightGBM)', ascending=False).reset_index(drop=True)

display(df_scores)
df_scores.to_csv("../output/scAnalyzer_LightGBM_nakatake_results.csv", index=False)

,Metric,Score (scAnalyzer_LightGBM)
0,bioplanet_2019_gs_n_active,973.0
1,hallmark_2020_gs_n_active,41.0
2,wikipathways_2019_gs_n_active,310.0
3,go_bp_2023_gs_n_active,2487.0
4,kegg_2021_gs_n_active,230.0
5,reactome_2022_gs_n_active,1143.0
6,hallmark_2020_gs_precision,0.8205128205128205
7,hallmark_2020_gs_f1,0.8
8,wikipathways_2019_gs_precision,0.7981651376146789
9,hallmark_2020_gs_recall,0.7804878048780488
